# 📊 Odin Daily Call Report Generator (Colab Edition) — **v2**

This interactive notebook wraps **`daily_odin_report.py`** and breaks it into easy‑to‑test units so you can:

* configure variables in one place  
* run each functional step independently  
* experiment with different report windows, batch sizes, etc.  

> **Important**: **Upload and import the script first** (see next section), then edit the configuration.


In [ ]:
# @@ Install / upgrade required libraries (Colab already has pandas/requests) @@
!pip -q install paramiko python-dateutil tqdm pandas requests ipywidgets

## Configuration

Edit the cell below to set **environment variables** or direct constants that the script relies on.


In [ ]:
# @title Configuration {"run":"auto"}

# @@ Runtime settings (edit freely) @@
import os, json, datetime
from datetime import timedelta
from dataclasses import asdict

# --- Minimum required ---
ODIN_API_BASE_URL = "" # @param {"type":"string"}
ODIN_API_USERNAME = "" # @param {"type":"string"}
ODIN_API_PASSWORD = "" # @param {"type":"string"}

SFTP_HOST = "" # @param {"type":"string"}
SFTP_USERNAME = "" # @param {"type":"string"}
SFTP_PASSWORD = "" # @param {"type":"string"}
SFTP_REMOTE_PATH = ""  # @param {"type":"string","placeholder":"/reports"}

SMTP_HOST = "" # @param {"type":"string"}
SMTP_USERNAME = "" # @param {"type":"string"}
SMTP_PASSWORD = "" # @param {"type":"string"}
SMTP_FROM = "" # @param {"type":"string","placeholder":"report@example.com"}
SMTP_TO = "" # @param {"type":"string","placeholder":"you@example.com"}
SMTP_PORT = "" # @param {"type":"string","placeholder":"587"}

start_date = "2025-07-01"  # e.g., "2025-07-01" or leave empty for yesterday
end_date = "2025-07-01"    # e.g., "2025-07-01" or leave empty for yesterday

# If dates are not specified, default to previous day
if not start_date or not end_date:
    yesterday = datetime.now() - timedelta(days=1)
    start_date = yesterday.strftime('%Y-%m-%d')
    end_date = yesterday.strftime('%Y-%m-%d')
    print(f"📅 Using default date range: {start_date} (previous day)")
else:
    print(f"📅 Using specified date range: {start_date} to {end_date}")

# Add time components
start_date_time = start_date + " 00:00:00"
end_date_time = end_date + " 23:59:59"

print(f"📊 Report Date Range: {start_date_time} to {end_date_time}")


# --- Minimum required ---
os.environ['ODIN_API_BASE_URL'] = ODIN_API_BASE_URL
os.environ['ODIN_API_USERNAME'] = ODIN_API_USERNAME
os.environ['ODIN_API_PASSWORD'] = ODIN_API_PASSWORD

# --- Optional overrides ---
os.environ['REPORT_START_DATE'] = start_date_time
os.environ['REPORT_END_DATE'] = end_date_time
os.environ['BATCH_SIZE'] = '200'

# SFTP (leave blank to disable)
os.environ['SFTP_HOST'] = SFTP_HOST
os.environ['SFTP_USERNAME'] = SFTP_USERNAME
os.environ['SFTP_PASSWORD'] = SFTP_PASSWORD

# SMTP (leave blank to disable)
os.environ['SMTP_HOST'] = SMTP_HOST
os.environ['SMTP_PORT'] = SMTP_PORT
os.environ['SMTP_USERNAME'] = SMTP_USERNAME
os.environ['SMTP_PASSWORD'] = SMTP_PASSWORD
os.environ['SMTP_FROM'] = SMTP_FROM
os.environ['SMTP_TO'] = SMTP_TO
os.environ['SFTP_REMOTE_PATH'] = SFTP_REMOTE_PATH

print("✅ Configuration Settings Loaded")

## 📥 Load `daily_odin_report.py`

* If running in Colab, upload the script when prompted below (first‑time run)  
* If the script lives on GitHub/GCS, you can `wget`/`gsutil cp` instead.


In [ ]:
# === Fresh-upload & import daily_odin_report.py ============================
import json
from dataclasses import asdict
import os, shutil, importlib.util, sys, re, textwrap
from google.colab import files  # pyright: ignore[reportMissingImports]

# 1️⃣  Make sure any old copy is removed so we *must* upload a new one
for fname in ("daily_odin_report.py",):
    if os.path.exists(fname):
        os.remove(fname)

# 2️⃣  Prompt user to upload a fresh file every run
uploaded = files.upload()                      # ⇧ choose your latest daily_odin_report.py
if not uploaded:
    raise FileNotFoundError("You must upload daily_odin_report.py to continue")

src_name = next(iter(uploaded))                # first (and usually only) uploaded file
shutil.move(src_name, "daily_odin_report.py")  # rename / overwrite

# 3️⃣  Hot-patch the mutable-default list issue, if still present
path = "daily_odin_report.py"
txt  = open(path).read()

if "smtp_to: List[str] =" in txt and "field(" not in txt:       # unpatched version
    # ensure 'field' is imported
    if "from dataclasses import" in txt:
        txt = re.sub(r"from dataclasses import ([^\n]+)",
                     lambda m: ("field" in m.group(1) and m.group(0) or
                                 f"from dataclasses import {m.group(1)}, field"),
                     txt, 1)
    else:
        txt = "from dataclasses import field\n" + txt

    # replace smtp_to line with default_factory version
    txt = re.sub(
        r"smtp_to\s*:\s*List\[str\]\s*=\s*[^\n]+",
        textwrap.dedent("""\
            smtp_to: List[str] = field(
                default_factory=lambda: [
                    a.strip() for a in os.getenv('SMTP_TO', '').split(',') if a.strip()
                ]
            )"""),
        txt, 1
    )
    open(path, "w").write(txt)
    print("🔧  Applied mutable-default patch")

# 4️⃣  Dynamic import
spec = importlib.util.spec_from_file_location("daily_odin_report", path)
dcr  = importlib.util.module_from_spec(spec)
sys.modules["daily_odin_report"] = dcr
spec.loader.exec_module(dcr)

print("✅  daily_odin_report.py uploaded & imported fresh")

## 🔑 Authenticate & Create API Client

In [ ]:
cfg = dcr.Config()
print(json.dumps(asdict(cfg), indent=2))
cfg = dcr.Config()                       # pick up env vars
api_client = dcr.OdinAPIClient(cfg)
assert api_client.authenticate(), "API authentication failed ❌"
print("Authenticated ✔")

## 🏢 Fetch All Service Providers

In [ ]:
# Get all Service Providers
service_providers = api_client.get_service_providers()
len(service_providers), service_providers[:5]

## 🏢 Fetch Service Providers from list

In [ ]:
# Get Service Providers from the list
SERVICE_PROVIDER_LIST = ["ent.odin"] # @param {"type":"string"}

all_sps = api_client.get_service_providers()
service_providers = [
    sp for sp in all_sps
    if sp.get('serviceProviderId') in SERVICE_PROVIDER_LIST
]
len(service_providers), service_providers[0] if service_providers else None

## Fetch Call Records for All Users (batched)

In [ ]:
from tqdm.auto import tqdm

start_date, end_date = dcr.DailyCallReportGenerator(cfg)._get_date_range()
all_call_records = []
total_users = 0
base = cfg.api_base_url.rstrip('/')

for sp in tqdm(service_providers, desc='Service Providers'):
    users    = api_client.get_users_for_service_provider(sp['serviceProviderId'])
    user_ids = [u['userId'] for u in users if u.get('userId')]
    total_users += len(user_ids)

    # 👉 One request per user
    for user_id in tqdm(user_ids, desc='Users', leave=False):
        resp = api_client.session.get(
            f"{base}/api/v2/users/call-records/details",
            params={
                'userIds':   [user_id],
                'startTime': start_date,
                'endTime':   end_date
            }
        )
        resp.raise_for_status()
        jr = resp.json()
        all_call_records.extend(jr.get('data', []))

print(f"Fetched {len(all_call_records):,} records for {total_users:,} users")

## 🗃️ Process & Aggregate Call Records Data

In [ ]:
## 🗃️ Process & Aggregate Call Records Data

# If you have run the "Fetch Call Records for All Users (batched)" cell and have all_call_records:
if 'all_call_records' in globals() and all_call_records:
    print("Processing and aggregating call records data...")
    processor = dcr.CallRecordProcessor(cfg)
    df_raw, df_agg = processor.process_call_records(all_call_records)
    print(f"Processed call records shape: {df_raw.shape}")
    print(f"Aggregated call records shape: {df_agg.shape}")
    if not df_agg.empty:
        print("\nAggregated Call Records by Service Provider:")
        print(df_agg.head())
else:
    print("No call records data found. Please run the call records fetch cell first.")

## 📊 Average Minutes Usage Summary

Calculates **average minutes usage** at three levels using `totalTime`/`totalSeconds` duration:
1. **By Group** - per groupId within each serviceProviderId
2. **By Service Provider** - aggregated across all groups per SP
3. **System-Wide** - overall totals across the entire system

**Both metrics included at each level:**
- **Answered calls only** - talk time for connected calls
- **All calls (answered + missed)** - total usage including unanswered

In [ ]:
# 📊 Average Minutes Usage Summary (answered + all calls)

import pandas as pd
import os
from datetime import datetime

if 'df_raw' not in globals() or df_raw.empty:
    print("⚠️ No call records data found. Run the 'Process & Aggregate' cell first.")
else:
    print(f"Processing {len(df_raw):,} total calls from {start_date} to {end_date} ...")
    
    # 1) Compute usage_seconds for ALL calls: prefer totalSeconds, fallback to parsing totalTime
    df_all = df_raw.copy()
    df_all["usage_seconds"] = pd.to_numeric(
        df_all.get("totalSeconds", 0), errors="coerce"
    ).fillna(0)

    if "totalTime" in df_all.columns:
        mask = (df_all["usage_seconds"] <= 0) & df_all["totalTime"].notna()
        df_all.loc[mask, "usage_seconds"] = (
            pd.to_timedelta(df_all.loc[mask, "totalTime"], errors="coerce")
            .dt.total_seconds().fillna(0)
        )
    
    df_all["groupId"] = df_all["groupId"].fillna("UNKNOWN")
    df_answered = df_all[df_all["answered"] == True].copy()
    print(f"Found {len(df_answered):,} answered calls (from {len(df_raw):,} total)")
    
    # ===== 1. BY GROUP =====
    df_all_grp = (
        df_all
        .groupby(["serviceProviderId", "groupId"], dropna=False)
        .agg(
            all_calls=("recordId", "count"),
            all_usage_minutes=("usage_seconds", lambda s: s.sum() / 60.0),
        )
        .reset_index()
    )
    
    if not df_answered.empty:
        df_ans_grp = (
            df_answered
            .groupby(["serviceProviderId", "groupId"], dropna=False)
            .agg(
                answered_calls=("recordId", "count"),
                answered_usage_minutes=("usage_seconds", lambda s: s.sum() / 60.0),
            )
            .reset_index()
        )
        df_avg_minutes_by_group = df_all_grp.merge(df_ans_grp, on=["serviceProviderId", "groupId"], how="left")
        df_avg_minutes_by_group["answered_calls"] = df_avg_minutes_by_group["answered_calls"].fillna(0).astype(int)
        df_avg_minutes_by_group["answered_usage_minutes"] = df_avg_minutes_by_group["answered_usage_minutes"].fillna(0)
    else:
        df_avg_minutes_by_group = df_all_grp.copy()
        df_avg_minutes_by_group["answered_calls"] = 0
        df_avg_minutes_by_group["answered_usage_minutes"] = 0.0

    df_avg_minutes_by_group["avg_minutes_per_answered_call"] = pd.to_numeric(
        df_avg_minutes_by_group["answered_usage_minutes"] / df_avg_minutes_by_group["answered_calls"].replace(0, pd.NA),
        errors='coerce'
    ).round(2)
    df_avg_minutes_by_group["avg_minutes_per_call"] = pd.to_numeric(
        df_avg_minutes_by_group["all_usage_minutes"] / df_avg_minutes_by_group["all_calls"].replace(0, pd.NA),
        errors='coerce'
    ).round(2)
    
    df_avg_minutes_by_group = df_avg_minutes_by_group[[
        "serviceProviderId", "groupId",
        "answered_calls", "answered_usage_minutes", "avg_minutes_per_answered_call",
        "all_calls", "all_usage_minutes", "avg_minutes_per_call"
    ]].sort_values(["serviceProviderId", "all_usage_minutes"], ascending=[True, False])

    # ===== 2. BY SERVICE PROVIDER =====
    df_all_sp = (
        df_all
        .groupby("serviceProviderId", dropna=False)
        .agg(
            total_groups=("groupId", "nunique"),
            all_calls=("recordId", "count"),
            all_usage_minutes=("usage_seconds", lambda s: s.sum() / 60.0),
        )
        .reset_index()
    )
    
    if not df_answered.empty:
        df_ans_sp = (
            df_answered
            .groupby("serviceProviderId", dropna=False)
            .agg(
                answered_calls=("recordId", "count"),
                answered_usage_minutes=("usage_seconds", lambda s: s.sum() / 60.0),
            )
            .reset_index()
        )
        df_avg_minutes_by_sp = df_all_sp.merge(df_ans_sp, on="serviceProviderId", how="left")
        df_avg_minutes_by_sp["answered_calls"] = df_avg_minutes_by_sp["answered_calls"].fillna(0).astype(int)
        df_avg_minutes_by_sp["answered_usage_minutes"] = df_avg_minutes_by_sp["answered_usage_minutes"].fillna(0)
    else:
        df_avg_minutes_by_sp = df_all_sp.copy()
        df_avg_minutes_by_sp["answered_calls"] = 0
        df_avg_minutes_by_sp["answered_usage_minutes"] = 0.0
    
    df_avg_minutes_by_sp["avg_minutes_per_answered_call"] = pd.to_numeric(
        df_avg_minutes_by_sp["answered_usage_minutes"] / df_avg_minutes_by_sp["answered_calls"].replace(0, pd.NA),
        errors='coerce'
    ).round(2)
    df_avg_minutes_by_sp["avg_minutes_per_call"] = pd.to_numeric(
        df_avg_minutes_by_sp["all_usage_minutes"] / df_avg_minutes_by_sp["all_calls"].replace(0, pd.NA),
        errors='coerce'
    ).round(2)
    
    df_avg_minutes_by_sp = df_avg_minutes_by_sp[[
        "serviceProviderId", "total_groups",
        "answered_calls", "answered_usage_minutes", "avg_minutes_per_answered_call",
        "all_calls", "all_usage_minutes", "avg_minutes_per_call"
    ]].sort_values("all_usage_minutes", ascending=False)

    # ===== 3. SYSTEM-WIDE SUMMARY =====
    total_sps = df_all["serviceProviderId"].nunique()
    total_groups = df_all["groupId"].nunique()
    all_calls_total = len(df_all)
    all_usage_min_total = df_all["usage_seconds"].sum() / 60.0
    answered_calls_total = len(df_answered) if not df_answered.empty else 0
    answered_usage_min_total = df_answered["usage_seconds"].sum() / 60.0 if not df_answered.empty else 0.0
    
    df_system_summary = pd.DataFrame([{
        "metric": "SYSTEM_TOTAL",
        "total_service_providers": total_sps,
        "total_groups": total_groups,
        "answered_calls": answered_calls_total,
        "answered_usage_minutes": round(answered_usage_min_total, 2),
        "avg_minutes_per_answered_call": round(answered_usage_min_total / answered_calls_total, 2) if answered_calls_total > 0 else None,
        "all_calls": all_calls_total,
        "all_usage_minutes": round(all_usage_min_total, 2),
        "avg_minutes_per_call": round(all_usage_min_total / all_calls_total, 2) if all_calls_total > 0 else None,
    }])

    # ===== DISPLAY RESULTS =====
    print(f"\n{'='*60}")
    print("📊 SYSTEM-WIDE SUMMARY")
    print(f"{'='*60}")
    display(df_system_summary)
    
    print(f"\n{'='*60}")
    print(f"📊 BY SERVICE PROVIDER ({len(df_avg_minutes_by_sp)} providers)")
    print(f"{'='*60}")
    display(df_avg_minutes_by_sp)
    
    print(f"\n{'='*60}")
    print(f"📊 BY GROUP ({len(df_avg_minutes_by_group)} groups)")
    print(f"{'='*60}")
    display(df_avg_minutes_by_group)


## 💾 Export Call Records Data to CSV

In [ ]:
# 💾 Export Call Records Data to CSV

import os
from datetime import datetime

# If you have run the "Process & Aggregate Call Records Data" cell and have df_raw and df_agg:
if 'df_raw' in globals() and 'df_agg' in globals():
    exporter = dcr.ReportExporter(cfg)
    raw_csv, agg_csv = exporter.export_to_csv(df_raw, df_agg)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_files = [raw_csv, agg_csv]
    
    # Export avg_minutes_by_group if available
    if 'df_avg_minutes_by_group' in globals() and not df_avg_minutes_by_group.empty:
        avg_grp_csv = os.path.join(cfg.output_dir, f"avg_minutes_by_group_{timestamp}.csv")
        df_avg_minutes_by_group.to_csv(avg_grp_csv, index=False)
        csv_files.append(avg_grp_csv)
        print(f"✅ Exported: {avg_grp_csv}")
    
    # Export avg_minutes_by_service_provider if available
    if 'df_avg_minutes_by_sp' in globals() and not df_avg_minutes_by_sp.empty:
        avg_sp_csv = os.path.join(cfg.output_dir, f"avg_minutes_by_service_provider_{timestamp}.csv")
        df_avg_minutes_by_sp.to_csv(avg_sp_csv, index=False)
        csv_files.append(avg_sp_csv)
        print(f"✅ Exported: {avg_sp_csv}")
    
    # Export system_summary if available
    if 'df_system_summary' in globals() and not df_system_summary.empty:
        sys_csv = os.path.join(cfg.output_dir, f"system_usage_summary_{timestamp}.csv")
        df_system_summary.to_csv(sys_csv, index=False)
        csv_files.append(sys_csv)
        print(f"✅ Exported: {sys_csv}")
    
    print(f"\n📁 All exported files ({len(csv_files)}):")
    for f in csv_files:
        print(f"   {f}")
else:
    print("No processed call records data found. Please run the call records processing cell first.")

## ☁️ Optional Outputs (SFTP Upload & Email)

In [ ]:
## ☁️ Optional Outputs (SFTP Upload & Email)

uploaded = False; emailed = False
if cfg.sftp_host and 'csv_files' in globals():
    uploaded = exporter.upload_to_sftp(csv_files)
if cfg.smtp_host and 'csv_files' in globals():
    summary = dcr.DailyCallReportGenerator(cfg)._calculate_summary_stats(
        df_raw, df_agg, len(service_providers), total_users
    )
    emailed = exporter.send_email_report(csv_files, summary)
print('SFTP:', uploaded, 'Email:', emailed)

---
### ✅ Notebook Complete
Generated 2025-06-27 21:03 UTC